# Lesson 5: Fine-tuning NER Models

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Understand when and why to fine-tune NER models
2. Prepare and format custom NER datasets
3. Fine-tune BERT for token classification
4. Evaluate models with proper NER metrics
5. Handle common training challenges

---

## 📚 Table of Contents

1. [When to Fine-tune](#1-when-to-fine-tune)
2. [Dataset Preparation](#2-dataset-preparation)
3. [Tokenization & Label Alignment](#3-tokenization--label-alignment)
4. [Training with Hugging Face Trainer](#4-training-with-hugging-face-trainer)
5. [Evaluation Metrics](#5-evaluation-metrics)
6. [Hyperparameter Tuning](#6-hyperparameter-tuning)
7. [Saving & Deploying Models](#7-saving--deploying-models)
8. [Further Reading](#8-further-reading)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q transformers datasets evaluate seqeval accelerate

In [ ]:
# Import libraries
import torch
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
import evaluate
import warnings
warnings.filterwarnings('ignore')

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")

---

## 1. When to Fine-tune

### Decision Matrix

| Scenario | Solution |
|----------|----------|
| General entities (PER, ORG, LOC) | Pre-trained models |
| Custom entities, no data | GLiNER (zero-shot) |
| Custom entities, have data | **Fine-tune** |
| Domain adaptation | **Fine-tune** |
| Maximum accuracy needed | **Fine-tune** |
| Low latency requirements | **Fine-tune** (distilled) |

### Benefits of Fine-tuning

1. **Domain Adaptation**: Learn domain-specific patterns and vocabulary
2. **Custom Entities**: Support any entity type you define
3. **Improved Accuracy**: Often 5-15% better than zero-shot
4. **Faster Inference**: Optimized for your specific task
5. **Consistent Behavior**: Predictable outputs for your use case

### How Much Data Do You Need?

| Data Amount | Expected Performance |
|-------------|---------------------|
| 50-100 examples | Basic functionality |
| 500-1000 examples | Good performance |
| 2000-5000 examples | Production-ready |
| 10000+ examples | State-of-the-art |

---

## 2. Dataset Preparation

### Standard NER Dataset Format

The standard format uses lists of tokens and BIO tags:

In [ ]:
# Load the WNUT-17 dataset (Emerging entities from social media)
dataset = load_dataset("wnut_17")

print("📊 WNUT-17 Dataset Overview\n")
print(f"Dataset splits: {list(dataset.keys())}")
print(f"\nTrain examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"Test examples: {len(dataset['test'])}")

print(f"\nFeatures: {dataset['train'].features}")

In [ ]:
# Examine the label set
label_names = dataset['train'].features['ner_tags'].feature.names

print("🏷️ NER Labels in WNUT-17:\n")
for i, name in enumerate(label_names):
    print(f"   {i}: {name}")

In [ ]:
# Look at a sample
sample = dataset['train'][0]

print("📝 Sample from WNUT-17:\n")
print(f"Tokens: {sample['tokens']}")
print(f"\nNER Tags (indices): {sample['ner_tags']}")
print(f"\nNER Tags (names): {[label_names[t] for t in sample['ner_tags']]}")

print("\n" + "-" * 50)
print(f"{'Token':<20} {'Tag'}")
print("-" * 30)
for token, tag in zip(sample['tokens'], sample['ner_tags']):
    print(f"{token:<20} {label_names[tag]}")

### Creating Custom Datasets

In [ ]:
# Example: Creating a custom dataset for programming language NER

custom_data = [
    {
        "tokens": ["Python", "is", "a", "programming", "language", "created", "by", "Guido", "van", "Rossum", "."],
        "ner_tags": [1, 0, 0, 0, 0, 0, 0, 3, 4, 4, 0]  # 1=B-LANG, 3=B-PERSON, 4=I-PERSON
    },
    {
        "tokens": ["JavaScript", "was", "developed", "at", "Netscape", "by", "Brendan", "Eich", "."],
        "ner_tags": [1, 0, 0, 0, 5, 0, 3, 4, 0]  # 5=B-ORG
    },
    {
        "tokens": ["Java", "was", "created", "by", "James", "Gosling", "at", "Sun", "Microsystems", "."],
        "ner_tags": [1, 0, 0, 0, 3, 4, 0, 5, 6, 0]  # 6=I-ORG
    },
    {
        "tokens": ["The", "Ruby", "language", "was", "designed", "by", "Yukihiro", "Matsumoto", "."],
        "ner_tags": [0, 1, 0, 0, 0, 0, 3, 4, 0]
    },
    {
        "tokens": ["Rust", "is", "developed", "by", "Mozilla", "."],
        "ner_tags": [1, 0, 0, 0, 5, 0]
    },
]

# Define label mapping
custom_label_names = ["O", "B-LANG", "I-LANG", "B-PERSON", "I-PERSON", "B-ORG", "I-ORG"]

# Create HuggingFace dataset
custom_dataset = Dataset.from_list(custom_data)

print("📊 Custom Dataset Created:\n")
print(f"Number of examples: {len(custom_dataset)}")
print(f"Labels: {custom_label_names}")

# Display a sample
print("\nSample:")
for token, tag in zip(custom_dataset[0]['tokens'], custom_dataset[0]['ner_tags']):
    print(f"   {token:<15} → {custom_label_names[tag]}")

In [ ]:
# Convert from span annotations to BIO format
def spans_to_bio(text, spans, label_names):
    """
    Convert span annotations to BIO format.
    
    Args:
        text: Original text string
        spans: List of (start, end, label) tuples
        label_names: List of entity type names
    
    Returns:
        tokens, tags as lists
    """
    # Simple whitespace tokenization
    tokens = text.split()
    tags = ['O'] * len(tokens)
    
    # Track character positions
    char_pos = 0
    token_spans = []
    
    for i, token in enumerate(tokens):
        start = text.find(token, char_pos)
        end = start + len(token)
        token_spans.append((start, end, i))
        char_pos = end
    
    # Match spans to tokens
    for span_start, span_end, label in spans:
        first_token = True
        for tok_start, tok_end, tok_idx in token_spans:
            # Check overlap
            if tok_start >= span_start and tok_end <= span_end:
                if first_token:
                    tags[tok_idx] = f'B-{label}'
                    first_token = False
                else:
                    tags[tok_idx] = f'I-{label}'
    
    return tokens, tags

# Example usage
text = "Tim Cook is the CEO of Apple Inc."
spans = [
    (0, 8, "PERSON"),   # "Tim Cook"
    (23, 33, "ORG")     # "Apple Inc."
]

tokens, tags = spans_to_bio(text, spans, ["PERSON", "ORG"])

print("📝 Span to BIO Conversion:\n")
print(f"Text: {text}")
print(f"Spans: {spans}")
print(f"\nResult:")
for token, tag in zip(tokens, tags):
    print(f"   {token:<15} → {tag}")

---

## 3. Tokenization & Label Alignment

### The Alignment Challenge

BERT uses subword tokenization, which splits words into pieces:

```
Word:     "Schwarzenegger"
Subwords: ["Schwarz", "##ene", "##gger"]

Word label: "B-PERSON"
Subword labels: ???
```

We need to align word-level labels with subword tokens.

In [ ]:
# Load tokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Label configuration for WNUT-17
label_names = dataset['train'].features['ner_tags'].feature.names
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}

print(f"📋 Label Mapping:\n")
for label, idx in label2id.items():
    print(f"   {label} → {idx}")

In [ ]:
# Tokenize and align labels
def tokenize_and_align_labels(examples):
    """
    Tokenize text and align labels with subword tokens.
    
    Strategy:
    - First subword gets the original label
    - Subsequent subwords get -100 (ignored in loss)
    - Special tokens ([CLS], [SEP]) get -100
    """
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,  # Input is already tokenized at word level
        max_length=128
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Special token
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a new word
                label_ids.append(label[word_idx])
            else:
                # Continuation subword
                label_ids.append(-100)
            
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized["labels"] = labels
    return tokenized

# Apply tokenization to dataset
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("✅ Dataset tokenized and labels aligned!")
print(f"\nTokenized dataset: {tokenized_dataset}")

In [ ]:
# Visualize the alignment
sample_idx = 0
original = dataset['train'][sample_idx]
tokenized = tokenized_dataset['train'][sample_idx]

print("🔗 Label Alignment Visualization:\n")
print("Original:")
print(f"  Tokens: {original['tokens']}")
print(f"  Labels: {[label_names[t] for t in original['ner_tags']]}")

print("\nTokenized (with alignment):")
tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'])
labels = tokenized['labels']

print(f"  {'Token':<15} {'Label ID':<10} {'Label'}")
print("  " + "-" * 40)
for token, label_id in zip(tokens, labels):
    if label_id == -100:
        label_str = "[IGNORED]"
    else:
        label_str = label_names[label_id]
    print(f"  {token:<15} {label_id:<10} {label_str}")

---

## 4. Training with Hugging Face Trainer

### Setting Up the Model

In [ ]:
# Load pre-trained model with classification head
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Model loaded: {model_checkpoint}")
print(f"   Number of labels: {len(label_names)}")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Data collator (handles padding)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Load seqeval metric
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    """
    Compute NER metrics using seqeval.
    """
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored labels (-100)
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("✅ Metrics function ready")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./ner-finetuned",
    
    # Training hyperparameters
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    
    # Evaluation strategy
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    
    # Logging
    logging_dir="./logs",
    logging_steps=100,
    
    # Other settings
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    report_to="none",  # Disable wandb/tensorboard
)

print("✅ Training arguments configured")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer created")

In [ ]:
# Train the model
print("🏋️ Starting training...\n")

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"\nTraining metrics:")
for key, value in train_result.metrics.items():
    print(f"   {key}: {value:.4f}")

In [ ]:
# Evaluate on test set
print("📊 Evaluating on test set...\n")

eval_results = trainer.evaluate(tokenized_dataset["test"])

print("Test Results:")
print(f"   Precision: {eval_results['eval_precision']:.4f}")
print(f"   Recall: {eval_results['eval_recall']:.4f}")
print(f"   F1 Score: {eval_results['eval_f1']:.4f}")
print(f"   Accuracy: {eval_results['eval_accuracy']:.4f}")

---

## 5. Evaluation Metrics

### Understanding NER Metrics

In [ ]:
# Detailed evaluation with per-entity metrics
def detailed_evaluation(trainer, dataset, label_names):
    """
    Perform detailed evaluation with per-entity type metrics.
    """
    predictions = trainer.predict(dataset)
    preds = np.argmax(predictions.predictions, axis=2)
    labels = predictions.label_ids
    
    # Convert to label names
    true_predictions = [
        [label_names[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]
    
    # Compute detailed metrics
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    
    return results

# Get detailed results
detailed_results = detailed_evaluation(trainer, tokenized_dataset["test"], label_names)

print("📊 Detailed Evaluation Results\n")
print("=" * 60)

# Overall metrics
print("\nOverall Metrics:")
print(f"   Precision: {detailed_results['overall_precision']:.4f}")
print(f"   Recall:    {detailed_results['overall_recall']:.4f}")
print(f"   F1:        {detailed_results['overall_f1']:.4f}")
print(f"   Accuracy:  {detailed_results['overall_accuracy']:.4f}")

# Per-entity metrics
print("\nPer-Entity Metrics:")
print(f"{'Entity':<15} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support'}")
print("-" * 60)

for entity_type in sorted(detailed_results.keys()):
    if entity_type.startswith('overall'):
        continue
    metrics = detailed_results[entity_type]
    print(f"{entity_type:<15} {metrics['precision']:<12.4f} {metrics['recall']:<12.4f} {metrics['f1']:<12.4f} {metrics['number']}")

### Confusion Analysis

In [ ]:
# Analyze common errors
from collections import Counter

def analyze_errors(trainer, dataset, tokenizer, label_names, num_examples=5):
    """
    Analyze prediction errors to understand model weaknesses.
    """
    predictions = trainer.predict(dataset)
    preds = np.argmax(predictions.predictions, axis=2)
    labels = predictions.label_ids
    
    # Count confusion types
    confusion = Counter()
    error_examples = []
    
    for i, (pred_seq, label_seq, input_ids) in enumerate(zip(preds, labels, dataset['input_ids'])):
        tokens = tokenizer.convert_ids_to_tokens(input_ids)
        
        for j, (p, l) in enumerate(zip(pred_seq, label_seq)):
            if l == -100:
                continue
            
            if p != l:
                pred_label = label_names[p]
                true_label = label_names[l]
                confusion[(true_label, pred_label)] += 1
                
                if len(error_examples) < num_examples:
                    error_examples.append({
                        'token': tokens[j],
                        'predicted': pred_label,
                        'actual': true_label,
                        'context': ' '.join(tokens[max(0, j-2):j+3])
                    })
    
    return confusion, error_examples

# Analyze errors
confusion, error_examples = analyze_errors(
    trainer, 
    tokenized_dataset["test"][:100],  # First 100 examples
    tokenizer, 
    label_names
)

print("🔍 Error Analysis\n")
print("Most Common Confusions:")
for (true, pred), count in confusion.most_common(10):
    print(f"   {true} → {pred}: {count}")

print("\nExample Errors:")
for err in error_examples[:5]:
    print(f"   Token: '{err['token']}'")
    print(f"   Predicted: {err['predicted']}, Actual: {err['actual']}")
    print(f"   Context: {err['context']}")
    print()

---

## 6. Hyperparameter Tuning

### Key Hyperparameters for NER

In [ ]:
# Hyperparameter search space
hyperparameters = {
    "learning_rate": [1e-5, 2e-5, 3e-5, 5e-5],
    "batch_size": [8, 16, 32],
    "epochs": [2, 3, 4, 5],
    "weight_decay": [0.0, 0.01, 0.1],
    "warmup_ratio": [0.0, 0.1],
}

print("🎛️ Key Hyperparameters for NER Fine-tuning\n")

for param, values in hyperparameters.items():
    print(f"{param}:")
    print(f"   Typical values: {values}")
    print()

In [ ]:
# Simple grid search example
def quick_train(lr, epochs, train_data, val_data):
    """
    Quick training function for hyperparameter search.
    """
    # Reload model
    model = AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id
    )
    
    args = TrainingArguments(
        output_dir="./ner-hp-search",
        learning_rate=lr,
        per_device_train_batch_size=16,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        logging_steps=1000,
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=val_data,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    results = trainer.evaluate()
    
    return results['eval_f1']

# Example: Compare learning rates (using small subset for speed)
print("🔬 Quick Hyperparameter Comparison\n")
print("Note: Using small subset for demonstration\n")

# Use small subset for quick demo
small_train = tokenized_dataset["train"].select(range(min(500, len(tokenized_dataset["train"]))))
small_val = tokenized_dataset["validation"].select(range(min(100, len(tokenized_dataset["validation"]))))

results_table = []
for lr in [2e-5, 5e-5]:
    f1 = quick_train(lr, 1, small_train, small_val)  # 1 epoch for speed
    results_table.append((lr, f1))
    print(f"   LR={lr}: F1={f1:.4f}")

best_lr = max(results_table, key=lambda x: x[1])
print(f"\n✅ Best learning rate: {best_lr[0]} (F1={best_lr[1]:.4f})")

---

## 7. Saving & Deploying Models

### Saving the Model

In [ ]:
# Save the fine-tuned model
output_dir = "./ner-finetuned-final"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to {output_dir}")

# List saved files
import os
print(f"\nSaved files:")
for f in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, f)) / 1024 / 1024
    print(f"   {f}: {size:.2f} MB")

In [ ]:
# Load and use the saved model
from transformers import pipeline

# Load as pipeline
ner_pipeline = pipeline(
    "ner",
    model=output_dir,
    aggregation_strategy="simple"
)

# Test inference
test_text = "The new iPhone was announced at Apple Park in Cupertino."

results = ner_pipeline(test_text)

print("🧪 Inference with Saved Model\n")
print(f"Text: {test_text}\n")
print("Entities:")
for entity in results:
    print(f"   {entity['word']:<20} → {entity['entity_group']} ({entity['score']:.4f})")

In [ ]:
# Create a production-ready inference function
class NERModel:
    """
    Production-ready NER model wrapper.
    """
    
    def __init__(self, model_path):
        self.pipeline = pipeline(
            "ner",
            model=model_path,
            aggregation_strategy="simple"
        )
    
    def extract(self, text, min_confidence=0.5):
        """
        Extract entities from text.
        
        Args:
            text: Input text
            min_confidence: Minimum confidence threshold
        
        Returns:
            List of entity dictionaries
        """
        if not text or not text.strip():
            return []
        
        results = self.pipeline(text)
        
        # Filter by confidence
        entities = [
            {
                "text": r["word"],
                "label": r["entity_group"],
                "confidence": r["score"],
                "start": r["start"],
                "end": r["end"]
            }
            for r in results
            if r["score"] >= min_confidence
        ]
        
        return entities
    
    def extract_batch(self, texts, min_confidence=0.5):
        """
        Extract entities from multiple texts.
        """
        all_results = self.pipeline(texts)
        
        batch_entities = []
        for results in all_results:
            entities = [
                {
                    "text": r["word"],
                    "label": r["entity_group"],
                    "confidence": r["score"]
                }
                for r in results
                if r["score"] >= min_confidence
            ]
            batch_entities.append(entities)
        
        return batch_entities

# Test the wrapper
ner = NERModel(output_dir)

texts = [
    "Google announced new AI features at their I/O conference.",
    "Tim Cook visited the Tesla factory in Austin."
]

print("🚀 Production Model Test\n")
for text in texts:
    entities = ner.extract(text)
    print(f"Text: {text}")
    print(f"Entities: {[(e['text'], e['label']) for e in entities]}\n")

---

## 8. Further Reading

### 📚 Essential Resources

1. **Hugging Face Token Classification Guide**
   - [Documentation](https://huggingface.co/docs/transformers/tasks/token_classification)

2. **Hugging Face Course - Fine-tuning for NER**
   - [Chapter 7](https://huggingface.co/learn/llm-course/en/chapter7/2)

3. **seqeval: Sequence Labeling Evaluation**
   - [GitHub](https://github.com/chakki-works/seqeval)

### 📖 Papers

1. **BERT for NER** (2019)
   - [Original BERT Paper](https://arxiv.org/abs/1810.04805)

2. **Named Entity Recognition is Not Solved** (2020)
   - [arXiv:2004.05107](https://arxiv.org/abs/2004.05107)

### 🎓 Tutorials

- [Fine-tuning BERT for NER - Blog Post](https://huggingface.co/blog/how-to-train)
- [Custom NER with Transformers](https://www.philschmid.de/fine-tune-bert-for-ner)

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **When to Fine-tune**: Decision matrix for approach selection
2. **Dataset Preparation**: BIO format and custom dataset creation
3. **Label Alignment**: Handling subword tokenization
4. **Training**: Using Hugging Face Trainer
5. **Evaluation**: seqeval metrics and error analysis
6. **Hyperparameter Tuning**: Key parameters to optimize
7. **Deployment**: Saving and using trained models

### 🚀 Next Lesson Preview

In **Lesson 6**, we'll explore **Advanced NER with NuNER**, including:
- Comparing zero-shot models (GLiNER vs NuNER)
- Few-shot NER techniques
- Model selection guidelines

In [ ]:
# Cleanup
import shutil

for dir_name in ["./ner-finetuned", "./ner-finetuned-final", "./ner-hp-search", "./logs"]:
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)
        print(f"🧹 Cleaned up {dir_name}")

print("\n🎉 Congratulations! You've completed Lesson 5: Fine-tuning NER Models")
print("\n📝 Key takeaways:")
print("   1. Fine-tuning gives best accuracy for specific domains")
print("   2. Label alignment with subwords is critical")
print("   3. Use seqeval for proper NER evaluation")
print("   4. Learning rate is the most important hyperparameter")
print("   5. Save both model and tokenizer for deployment")
print("\n👉 Continue to Lesson 6: Advanced NER with NuNER")